EXP 5 RETRIEVAL AUGMENTED

In [5]:
!pip install -q sentence-transformers faiss-cpu transformers torch accelerate

In [1]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

In [2]:
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

In [3]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
doc_embeddings = embed_model.encode(documents)

In [5]:
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(doc_embeddings))

In [7]:
query = "What is RAG in AI?"

In [8]:
query_embedding = embed_model.encode([query])

In [9]:
D, I = index.search(np.array(query_embedding), k=2)

retrieved_chunks = [documents[i] for i in I[0]]

In [10]:
context = " ".join(retrieved_chunks)

prompt = f"""
Context:
{context}

Question:
{query}

Answer:
"""

In [11]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [12]:
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=60
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

In [13]:
print("Retrieved Context:\n")

for chunk in retrieved_chunks:
    print("-", chunk)

print("\nGenerated Answer:\n")

print(answer)

Retrieved Context:

- Python is a popular high-level programming language used in AI development.
- Retrieval-Augmented Generation combines document retrieval with text generation.

Generated Answer:

combines document retrieval with text generation
